# 01 -- Sphere scattering validation

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/antnewman/acoustic-beacon-optimiser/blob/main/notebooks/01_sphere_validation.ipynb)

Validate the BEM solver against the analytical Mie series solution for a rigid sphere.
This notebook regenerates the Mie-vs-BEM comparison reported in the paper.

**What this shows:** the bempp-cl Burton-Miller CFIE solver agrees with the exact analytical solution to within approximately 0.03 dB at Helmholtz numbers He = 5 and He = 10 when the mesh is adequately refined (roughly 10 elements per wavelength).

**Runtime:** a few minutes per case on a single CPU.

**Setup in Colab:** run `!pip install -q git+https://github.com/antnewman/acoustic-beacon-optimiser.git` in the first cell, then continue.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

import bempp_cl.api as bempp
from abo.acoustics.bem_solver import scattered_field, solve_scattering
from tests.analytical.mie_series import mie_backscatter_ts

## Configuration

In [ ]:
SPHERE_RADIUS = 1.0
SPEED_OF_SOUND = 343.0
FAR_FIELD_RANGE = 10.0

# Helmholtz numbers to validate and corresponding bempp refinement levels
CASES = [(5, 4), (10, 5), (20, 6)]

def frequency_for_he(he):
    return he * SPEED_OF_SOUND / (2.0 * np.pi * SPHERE_RADIUS)

## Run BEM and Mie for each case

In [ ]:
results = []
for he, level in CASES:
    f = frequency_for_he(he)
    grid = bempp.shapes.regular_sphere(level)
    direction = np.array([0.0, 0.0, -1.0])
    total = solve_scattering(grid, f, direction, SPEED_OF_SOUND)
    pt = np.array([[0.0], [0.0], [FAR_FIELD_RANGE]])
    p_scat = scattered_field(grid, total, pt, f, direction, SPEED_OF_SOUND)
    ts_bem = 10.0 * np.log10(np.abs(p_scat[0]) ** 2 * FAR_FIELD_RANGE**2)
    ts_mie = mie_backscatter_ts(SPHERE_RADIUS, f, SPEED_OF_SOUND, FAR_FIELD_RANGE)
    results.append({
        "he": he,
        "level": level,
        "n_elements": grid.number_of_elements,
        "ts_bem": float(ts_bem),
        "ts_mie": float(ts_mie),
        "error_db": float(abs(ts_bem - ts_mie)),
    })
    print(f"He={he}: BEM={ts_bem:.3f} dB, Mie={ts_mie:.3f} dB, err={abs(ts_bem-ts_mie):.3f} dB")

## Plot the error vs He

In [ ]:
hes = np.array([r["he"] for r in results])
errs = np.array([r["error_db"] for r in results])

fig, ax = plt.subplots(figsize=(7, 4))
ax.semilogy(hes, errs, "o-", linewidth=1.5)
ax.axhline(0.5, linestyle="--", linewidth=1.0, label="0.5 dB tolerance")
ax.set_xlabel("Helmholtz number He = ka")
ax.set_ylabel("|BEM - Mie| (dB)")
ax.set_title("BEM-vs-Mie backscatter TS agreement, rigid sphere")
ax.grid(visible=True, linestyle=":", linewidth=0.5)
ax.legend()
fig.tight_layout()
fig.savefig("../paper/figures/fig01_mie_validation.png", dpi=200)
plt.show()